- In this notebook, we will dive into the data on which our project will be made.
- I got this data from GitHubgithub.com/chatterjeesaurabh/Credit-Risk-Modelling-using-Machine-Learning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df_1 = pd.read_excel("D:/Study/IFRS-9-Complient-Risk-Analysis-modelling/data/case_study1.xlsx")
df_2 = pd.read_excel("D:/Study/IFRS-9-Complient-Risk-Analysis-modelling/data/case_study2.xlsx")

In [ ]:
# Let us save these datasets into csv format for faster retrieval

df_1.to_csv('D:/Study/IFRS-9-Complient-Risk-Analysis-modelling/data/01_dataset_raw.csv')
df_2.to_csv('D:/Study/IFRS-9-Complient-Risk-Analysis-modelling/data/02_dataset_raw.csv')

### Overview

In [18]:
# Basic Dataset Information

print(f'Dataset 1 -> {df_1.shape}')
print(f'Dataset 2 -> {df_2.shape}')

Dataset 1 -> (51336, 26)
Dataset 2 -> (51336, 62)


- Dataset 1 Primarily consists of internal bureau data, aggregating the customer's trade lines (loans), historical loan counts, and payment behavior (active, closed, missed payments).

- Dataset 2 Contains detailed delinquency history, recent inquiries, demographic information (Age, Income, Education), and the target risk variable.


In [9]:
df_1.head()

,PROSPECTID,Total_TL,Tot_Closed_TL,Tot_Active_TL,Total_TL_opened_L6M,Tot_TL_closed_L6M,pct_tl_open_L6M,pct_tl_closed_L6M,pct_active_tl,pct_closed_tl,...,CC_TL,Consumer_TL,Gold_TL,Home_TL,PL_TL,Secured_TL,Unsecured_TL,Other_TL,Age_Oldest_TL,Age_Newest_TL
0,1,5,4,1,0,0,0.000,0.0,0.200,0.800,...,0,0,1,0,4,1,4,0,72,18
1,2,1,0,1,0,0,0.000,0.0,1.000,0.000,...,0,1,0,0,0,0,1,0,7,7
2,3,8,0,8,1,0,0.125,0.0,1.000,0.000,...,0,6,1,0,0,2,6,0,47,2
3,4,1,0,1,1,0,1.000,0.0,1.000,0.000,...,0,0,0,0,0,0,1,1,5,5
4,5,3,2,1,0,0,0.000,0.0,0.333,0.667,...,0,0,0,0,0,3,0,2,131,32


In [10]:
df_2.head()

,PROSPECTID,time_since_recent_payment,time_since_first_deliquency,time_since_recent_deliquency,num_times_delinquent,max_delinquency_level,max_recent_level_of_deliq,num_deliq_6mts,num_deliq_12mts,num_deliq_6_12mts,...,pct_CC_enq_L6m_of_L12m,pct_PL_enq_L6m_of_ever,pct_CC_enq_L6m_of_ever,max_unsec_exposure_inPct,HL_Flag,GL_Flag,last_prod_enq2,first_prod_enq2,Credit_Score,Approved_Flag
0,1,549,35,15,11,29,29,0,0,0,...,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,47,-99999,-99999,0,-99999,0,0,0,0,...,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,302,11,3,9,25,25,1,9,8,...,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2
3,4,-99999,-99999,-99999,0,-99999,0,0,0,0,...,0.0,0.0,0.0,9.900,0,0,others,others,673,P2
4,5,583,-99999,-99999,0,-99999,0,0,0,0,...,0.0,0.0,0.0,-99999.000,0,0,AL,AL,753,P1


In these datasets, `-99999` is used extensively as a missing value indicator.**
- Thus before performing any EDA or correlation checks, we will have to replace `-99999` with `np.nan`

- Both the datasets contain `PROSPECTID` Column which will help join the two datasets together into one dataset

In [12]:
merged_df = pd.merge(df_1 , df_2 , on = 'PROSPECTID' , how = 'inner')

In [15]:
# Overview of merged dataframe
print(merged_df.shape)

merged_df.head()

(51336, 87)


,PROSPECTID,Total_TL,Tot_Closed_TL,Tot_Active_TL,Total_TL_opened_L6M,Tot_TL_closed_L6M,pct_tl_open_L6M,pct_tl_closed_L6M,pct_active_tl,pct_closed_tl,...,pct_CC_enq_L6m_of_L12m,pct_PL_enq_L6m_of_ever,pct_CC_enq_L6m_of_ever,max_unsec_exposure_inPct,HL_Flag,GL_Flag,last_prod_enq2,first_prod_enq2,Credit_Score,Approved_Flag
0,1,5,4,1,0,0,0.000,0.0,0.200,0.800,...,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,1,0,1,0,0,0.000,0.0,1.000,0.000,...,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,8,0,8,1,0,0.125,0.0,1.000,0.000,...,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2
3,4,1,0,1,1,0,1.000,0.0,1.000,0.000,...,0.0,0.0,0.0,9.900,0,0,others,others,673,P2
4,5,3,2,1,0,0,0.000,0.0,0.333,0.667,...,0.0,0.0,0.0,-99999.000,0,0,AL,AL,753,P1


In [17]:
# Let us save this merged dataframe into csv file, for further investigation

merged_df.to_csv('D:/Study/IFRS-9-Complient-Risk-Analysis-modelling/data/merged_dataset.csv')

**Let us replace the values -99999 with NAN and then check the missing values percentage on each columns**

In [ ]:
round((merged_df == -99999).sum()/merged_df.shape[0]*100,2)
# This shows the percentage of values/data which is null in each column

PROSPECTID             0.0
Total_TL               0.0
Tot_Closed_TL          0.0
Tot_Active_TL          0.0
Total_TL_opened_L6M    0.0
                      ... 
GL_Flag                0.0
last_prod_enq2         0.0
first_prod_enq2        0.0
Credit_Score           0.0
Approved_Flag          0.0
Length: 87, dtype: float64

In [ ]:
merged_df.replace(-99999,np.nan)